# Where can you actually rent an H100?

GPU price lists tell you what a cloud **charges**. They don't tell you whether you can **get one**.

This notebook uses the [GPU Cloud Prices and Availability](https://www.kaggle.com/datasets/maxhager/gpu-cloud-prices) dataset, which checks every single-GPU offer at 59 clouds every 15 minutes and records both the price and whether it is actually rentable. Questions:

1. Which GPUs are listed by many clouds but rentable at few?
2. How often is an H100 actually in stock, cloud by cloud?
3. Is the cheapest *advertised* H100 the one you can rent?
4. Data center vs. marketplace prices
5. The GPU price index over time

All numbers below are computed from the data, so they update with each new version of the dataset.

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

base = os.path.dirname(glob.glob("/kaggle/input/**/daily_prices.csv", recursive=True)[0])
daily = pd.read_csv(f"{base}/daily_prices.csv", parse_dates=["date"])
offers = pd.read_csv(f"{base}/offers.csv")
clouds = pd.read_csv(f"{base}/clouds.csv")
gpus = pd.read_csv(f"{base}/gpus.csv")
index = pd.read_csv(f"{base}/index_daily.csv", parse_dates=["date"])

daily = daily[daily.date > daily.date.min()]          # the first day is partial
name = clouds.set_index("id").name.to_dict()
gpu_name = gpus.set_index("id").name.to_dict()

plt.rcParams.update({"figure.figsize": (11, 4.8), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 11})
INK, MUTED = "#1f2937", "#9ca3af"

print(f"{offers.provider_id.nunique()} clouds, {offers.gpu_id.nunique()} GPU models, {len(offers):,} offers")
print(f"daily data: {daily.date.min():%d %b} to {daily.date.max():%d %b %Y}")

## 1. Listed vs. rentable right now

For each GPU: how many clouds list it on demand, and how many have one free at this moment (`offers.csv`, latest snapshot).

In [ ]:
od = offers[(offers.pricing == "on_demand") & (offers.fresh == True)]
listed = od.groupby("gpu_id").provider_id.nunique()
rentable = od[od.in_stock == True].groupby("gpu_id").provider_id.nunique()
t = pd.DataFrame({"listed": listed, "rentable": rentable}).fillna(0).astype(int)
t = t.sort_values("listed", ascending=False).head(15)
t.index = [gpu_name.get(g, g) for g in t.index]

ax = t[::-1].plot.barh(color=[MUTED, INK], width=0.75, figsize=(11, 6))
ax.set_xlabel("clouds"); ax.set_title("Clouds listing each GPU vs. clouds where it is rentable right now")
ax.legend(["listed", "rentable now"], frameon=False)
plt.tight_layout(); plt.show()

t["share rentable"] = (t.rentable / t.listed).map("{:.0%}".format)
t

## 2. How often is an H100 SXM actually in stock?

`daily_prices.in_stock_share` is the share of each day an offer was rentable. Averaged over all days, per cloud (on-demand, single GPU):

In [ ]:
h = daily[(daily.gpu_id == "h100-sxm-80gb") & (daily.pricing == "on_demand")]
by_cloud = (h.groupby("provider_id")
              .agg(availability=("in_stock_share", "mean"),
                   avg_price=("price_avg", "mean"),
                   list_price=("list_price_per_hour", "last"))
              .sort_values("availability"))
by_cloud.index = [name.get(c, c) for c in by_cloud.index]

ax = by_cloud.availability.mul(100).plot.barh(color=INK, width=0.7, figsize=(11, 0.38 * len(by_cloud) + 1.5))
ax.set_xlabel("% of the time rentable"); ax.set_xlim(0, 100)
ax.set_title("H100 SXM, on-demand: share of time a single GPU was rentable")
plt.tight_layout(); plt.show()

always = (by_cloud.availability > 0.95).sum()
print(f"{len(by_cloud)} clouds list an on-demand H100 SXM; {always} had one rentable more than 95% of the time.")

### Price vs. availability

Is the cheap H100 the one that's always sold out? Each dot is a cloud: advertised price vs. how often it was rentable.

In [ ]:
d = by_cloud.dropna(subset=["list_price"])
fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(d.list_price, d.availability * 100, s=60, color=INK)
for n, r in d.iterrows():
    ax.annotate(n, (r.list_price, r.availability * 100), xytext=(6, 3), textcoords="offset points", fontsize=9)
ax.set_xlabel("advertised price, USD per GPU-hour"); ax.set_ylabel("% of the time rentable")
ax.set_title("H100 SXM: advertised price vs. availability")
plt.tight_layout(); plt.show()

## 3. Cheapest advertised vs. cheapest you could actually rent

On marketplaces the listed price *is* a free machine, so this only makes sense for **data-center** clouds. Per day: the lowest advertised on-demand H100 SXM price, and the lowest price that was actually rentable at some point that day.

In [ ]:
dc = h[h.cloud_type == "datacenter"]
rows = []
for day, g in dc.groupby("date"):
    adv = g.loc[g.list_price_per_hour.idxmin()]
    got = g.dropna(subset=["price_min"])
    got = got.loc[got.price_min.idxmin()] if len(got) else None
    rows.append({"date": day.date(), "cheapest advertised": adv.list_price_per_hour,
                 "at": name.get(adv.provider_id, adv.provider_id), "in stock that day": f"{adv.in_stock_share:.0%}",
                 "cheapest rentable": None if got is None else got.price_min,
                 "rentable at": None if got is None else name.get(got.provider_id, got.provider_id)})
cheap = pd.DataFrame(rows).set_index("date")
ax = cheap[["cheapest advertised", "cheapest rentable"]].plot(color=[MUTED, INK], lw=2, marker="o")
ax.set_ylabel("USD per GPU-hour"); ax.set_xlabel("")
ax.set_title("H100 SXM on-demand, data-center clouds: advertised vs. rentable")
plt.tight_layout(); plt.show()
cheap

## 4. Data center vs. marketplace

`cloud_type` separates clouds' own data centers from marketplaces where hosts list their own machines (Vast.ai, Clore, ...). Median of each cloud's daily average rentable price:

In [ ]:
popular = ["h100-sxm-80gb", "h200-sxm-141gb", "a100-sxm-80gb", "rtx-5090-32gb", "rtx-4090-24gb", "l40s-48gb"]
p = (daily[(daily.pricing == "on_demand") & daily.gpu_id.isin(popular)]
       .groupby(["gpu_id", "cloud_type"]).price_avg.median().unstack())
p.index = [gpu_name.get(g, g) for g in p.index]
ax = p.plot.bar(color=[INK, MUTED], rot=0, width=0.75)
ax.set_ylabel("USD per GPU-hour"); ax.set_title("Median rentable price: data center vs. marketplace")
ax.legend(frameon=False); plt.tight_layout(); plt.show()
p.round(2)

## 5. GPU Price Index

The [United Compute GPU Price Index](https://unitedcompute.ai/gpu-price-index): the median across clouds of each cloud's cheapest rentable on-demand price (resellers excluded).

In [ ]:
ix = index.pivot(index="date", columns="ticker", values="close")
ax = ix.plot(lw=2, marker="o", figsize=(11, 5))
ax.set_ylabel("USD per GPU-hour"); ax.set_xlabel(""); ax.set_title("GPU Price Index, daily close")
ax.legend(frameon=False, ncol=5); plt.tight_layout(); plt.show()
ix.round(2)

## Takeaways

In [ ]:
last = t.copy()
gap = cheap.iloc[-1]
mk = p.dropna()
print(f"- Of the {len(t)} most widely listed GPUs, the median one is listed by {int(t.listed.median())} clouds "
      f"but rentable right now at {int(t.rentable.median())}.")
print(f"- H100 SXM: {len(by_cloud)} clouds list it; {always} had one rentable more than 95% of the time.")
print(f"- Cheapest advertised data-center H100 on {cheap.index[-1]:%d %b}: ${gap['cheapest advertised']:.2f} at {gap['at']} "
      f"(in stock {gap['in stock that day']} of the day); cheapest you could actually rent: "
      f"${gap['cheapest rentable']:.2f} at {gap['rentable at']}.")
print(f"- Marketplaces were cheaper than data-center clouds for {(mk.community < mk.datacenter).sum()} of {len(mk)} popular GPUs.")

**Live version:** [unitedcompute.ai/gpu-price-tracker](https://unitedcompute.ai/gpu-price-tracker), updated every 15 minutes, with a free [API and MCP server](https://unitedcompute.ai/openapi.json). Same data on [Hugging Face](https://huggingface.co/datasets/united-compute/gpu-prices) and [GitHub](https://github.com/United-Compute/gpu-price-tracker).

Data: United Compute GPU Price Tracker, CC BY 4.0.